# 🍜 Fine-Tuning Llama 2 & Mistral 7B with [Soup](https://github.com/MakazhanAlpamys/Soup)

Uses the **Soup CLI** (`soup-cli`) to QLoRA fine-tune **Llama-2-7B** and **Mistral-7B** on a niche domain (default: agriculture Q&A), then uses `soup adapters merge` to blend several runs into one adapter (a "model soup"), and compares everything on held-out data.

**Pipeline**
1. Prepare a domain dataset as Alpaca-format JSONL
2. Generate a `soup.yaml` per run (different LR / seed) and train with `soup train`
3. Merge the adapters with `soup adapters merge` (uniform soup + greedy soup)
4. Evaluate base vs. single runs vs. soups
5. Chat / push with `soup chat` and `soup push`

> Llama 2 and Mistral have different architectures, so each family gets its own soup.
> Llama 2 is gated: accept the license on Hugging Face first. Soup is under fast development, so if a flag below errors, run `soup <command> --help` and adjust.

## 1. Install

In [ ]:
!pip install -q "soup-cli[train]" pandas pyyaml
!soup doctor

## 2. Configuration

In [ ]:
import os, json, glob, gc, math, subprocess, yaml, random
from pathlib import Path

MODELS = {
    "mistral": "mistralai/Mistral-7B-v0.1",
    "llama2":  "meta-llama/Llama-2-7b-hf",   # gated
}
RUN_MODELS = ["mistral", "llama2"]

# Domain dataset: swap for your own niche domain
DATASET_NAME = "KisanVaani/agriculture-qa-english-only"
QUESTION_COL, ANSWER_COL = "question", "answers"
DOMAIN = "agriculture"
MAX_SAMPLES = 5000

# Each recipe is one soup ingredient
RECIPES = [
    dict(name="r1", lr=1e-4, seed=1),
    dict(name="r2", lr=2e-4, seed=2),
    dict(name="r3", lr=3e-4, seed=3),
]
EPOCHS, LORA_R, LORA_ALPHA = 1, 16, 32

WORK = Path("work"); WORK.mkdir(exist_ok=True)

## 3. Hugging Face login

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

## 4. Prepare the dataset (Alpaca JSONL, which Soup auto-detects)

In [ ]:
from datasets import load_dataset

raw = load_dataset(DATASET_NAME, split="train")
print(raw); print(raw[0])
raw = raw.shuffle(seed=42).select(range(min(MAX_SAMPLES, len(raw))))

rows = [{"instruction": r[QUESTION_COL], "input": "", "output": str(r[ANSWER_COL])} for r in raw]
n = len(rows)
train_rows, val_rows, test_rows = rows[: int(n*.9)], rows[int(n*.9): int(n*.95)], rows[int(n*.95):]

def dump(path, data):
    with open(path, "w") as f:
        for r in data: f.write(json.dumps(r, ensure_ascii=False) + "\n")

dump(WORK/"train.jsonl", train_rows)   # given to soup (it also holds out a small val split)
dump(WORK/"val.jsonl",  val_rows)      # used for greedy-soup selection
dump(WORK/"test.jsonl", test_rows)     # final report only
print(len(train_rows), len(val_rows), len(test_rows))
!soup data inspect work/train.jsonl

## 5. Train every ingredient with `soup train`

In [ ]:
def write_config(model_key, rec):
    out = WORK / model_key / rec["name"]
    cfg = {
        "base": MODELS[model_key],
        "task": "sft",
        "data": {"train": str(WORK/"train.jsonl"), "format": "alpaca", "val_split": 0.05},
        "training": {
            "epochs": EPOCHS, "lr": rec["lr"], "batch_size": "auto", "seed": rec["seed"],
            "lora": {"r": LORA_R, "alpha": LORA_ALPHA},
            "quantization": "4bit",
        },
        "output": str(out),
    }
    path = WORK / f"{model_key}_{rec['name']}.yaml"
    path.write_text(yaml.safe_dump(cfg, sort_keys=False))
    return path, out

def adapter_dir(out):
    hits = glob.glob(f"{out}/**/adapter_config.json", recursive=True)
    assert hits, f"no adapter found under {out}"
    return str(Path(hits[0]).parent)

adapters = {k: {} for k in RUN_MODELS}
for key in RUN_MODELS:
    for rec in RECIPES:
        cfg, out = write_config(key, rec)
        if not glob.glob(f"{out}/**/adapter_config.json", recursive=True):
            print(f"\n=== training {key}/{rec['name']} ===")
            subprocess.run(["soup", "train", "--config", str(cfg)], check=True)
        adapters[key][rec["name"]] = adapter_dir(out)
adapters

## 6. Evaluation helper
Answer-only loss/perplexity on held-out rows, applied identically to the base model and every adapter.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
BNB = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=DTYPE)

def fmt(q): return f"### Instruction:\n{q}\n\n### Response:\n"

def load_rows(path): return [json.loads(l) for l in open(path)]

@torch.no_grad()
def answer_loss(model, tok, rows, n=200, max_len=512):
    model.eval(); tot, cnt = 0.0, 0
    for r in rows[:n]:
        p = tok(fmt(r["instruction"]), add_special_tokens=True)["input_ids"]
        a = tok(r["output"] + tok.eos_token, add_special_tokens=False)["input_ids"]
        ids = (p + a)[:max_len]
        labels = ([-100]*len(p) + a)[:max_len]
        x = torch.tensor([ids]).to(model.device); y = torch.tensor([labels]).to(model.device)
        loss = model(input_ids=x, labels=y).loss.item()
        k = sum(l != -100 for l in labels)
        tot += loss * k; cnt += k
    return tot / max(cnt, 1)

def eval_adapter(model_key, adapter_path, rows):
    tok = AutoTokenizer.from_pretrained(MODELS[model_key])
    base = AutoModelForCausalLM.from_pretrained(MODELS[model_key], quantization_config=BNB,
                                                device_map={"": 0}, torch_dtype=DTYPE)
    model = PeftModel.from_pretrained(base, adapter_path) if adapter_path else base
    loss = answer_loss(model, tok, rows)
    del model, base; gc.collect(); torch.cuda.empty_cache()
    return loss

val, test = load_rows(WORK/"val.jsonl"), load_rows(WORK/"test.jsonl")

## 7. Make the soups with `soup adapters merge`

* **Uniform soup**: merge all ingredients with the `linear` strategy.
* **Greedy soup**: rank ingredients by validation loss, then add each one only if the merged adapter's validation loss improves (Wortsman et al., 2022).

Other strategies are available (`ties`, `dare`, `svd`): change `STRATEGY` to try them.

In [ ]:
STRATEGY = "linear"

def soup_merge(paths, out, strategy=STRATEGY):
    out = str(out)
    if not glob.glob(f"{out}/adapter_config.json"):
        subprocess.run(["soup", "adapters", "merge", *paths, "-o", out, "--strategy", strategy], check=True)
    return adapter_dir(out)

results = {}
for key in RUN_MODELS:
    res = {"base": eval_adapter(key, None, val)}
    for name, path in adapters[key].items():
        res[name] = eval_adapter(key, path, val)
    ranked = sorted(adapters[key], key=lambda n: res[n])
    print(key, {k: round(v, 4) for k, v in res.items()})

    # Uniform soup
    uni = soup_merge(list(adapters[key].values()), WORK/key/"soup_uniform")
    res["uniform_soup"] = eval_adapter(key, uni, val)

    # Greedy soup
    chosen, best = [ranked[0]], res[ranked[0]]
    for cand in ranked[1:]:
        trial = chosen + [cand]
        tpath = soup_merge([adapters[key][n] for n in trial], WORK/key/("trial_" + "_".join(trial)))
        score = eval_adapter(key, tpath, val)
        print(f"  {'+' if score < best else '-'} {cand}: {score:.4f}")
        if score < best: best, chosen = score, trial
    greedy = soup_merge([adapters[key][n] for n in chosen], WORK/key/"soup_greedy")
    res["greedy_soup"], res["greedy_members"] = eval_adapter(key, greedy, val), chosen
    results[key] = res

json.dump(results, open(WORK/"val_results.json", "w"), indent=2)

## 8. Final comparison on the held-out test set

In [ ]:
import pandas as pd
rows = []
for key in RUN_MODELS:
    variants = {"base": None, **adapters[key],
                "uniform_soup": adapter_dir(WORK/key/"soup_uniform"),
                "greedy_soup":  adapter_dir(WORK/key/"soup_greedy")}
    for v, p in variants.items():
        rows.append((key, v, math.exp(eval_adapter(key, p, test))))
df = pd.DataFrame(rows, columns=["model", "variant", "test_perplexity"])
df.pivot(index="variant", columns="model", values="test_perplexity").round(3)

## 9. Try it, push it, merge it

In [ ]:
# Chat with the greedy soup (interactive): run in a terminal or a notebook cell
# !soup chat --model work/mistral/soup_greedy

# Push the adapter to the Hub
# !soup push --model work/mistral/soup_greedy --repo YOUR_USER/mistral-agri-soup

# Bake the adapter into full weights
# !soup merge --adapter work/mistral/soup_greedy

# Export for Ollama / llama.cpp
# !soup export --model work/mistral/soup_greedy --format gguf --quant q4_k_m

## Optional: low-VRAM training
Soup's *layer streaming* can train 7–8B models on ~4 GB GPUs. See the Soup docs for the exact `soup.yaml` setting (`soup train --help`, `docs/backends-and-ops.md`) and add it to `write_config` above.